In [2]:
import json
import numpy as np
import pandas as pd
import warnings
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
import xgboost as xgb
import lightgbm as lgb
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)

warnings.filterwarnings("ignore")

In [3]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_PATH  = r"D:\2026\MockProject_062026_NhomAI\data\healthcare_5cores.json"        
TARGET_COL = "Level"            
CORES_KEY  = "5_cores"          
TEST_SIZE  = 0.2                
VAL_SIZE   = 0.15              
MODEL_SAVE_DIR = r"D:\2026\MockProject_062026_NhomAI\model"  


In [4]:
def score_to_level(score):
    if score < 1.5:
        return 1
    elif score < 2.5:
        return 2
    return 3

In [5]:
print("=" * 60)
print("  LOAD & PREPROCESS DATA (REGRESSION & ORDINAL LEVEL)")
print("=" * 60)

with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

CORE_FEATURES = {
    "ADLs & IADLs": ["Age", "Physical Activity"],
    "Cognitive & Neurological Status": ["Sleep Duration", "Stress Level"],
    "Clinical Risk Assessments": ["BMI", "Chronic Disease"],
    "Mood & Behavioral Health": ["Stress Level", "Smoking Status", "Alcohol Consumption"],
    "Financial & Legal": ["Age", "Gender", "Chronic Disease"]
}

X_raw = {core_name: [] for core_name in CORE_FEATURES.keys()}
y_raw = []

for record in raw_data:
    cores = record.get("5_cores", record.get("5 cores", {}))
    for core_name, feat_list in CORE_FEATURES.items():
        core_data = cores.get(core_name, {})
        row = []
        for feat in feat_list:
            val = core_data.get(feat, 0)
            if val is None:
                if feat in ["Chronic Disease", "Smoking Status", "Gender"]:
                    val = "Unknown"
                else:
                    val = 0.0
            row.append(val)
        X_raw[core_name].append(row)
    y_raw.append(record.get(TARGET_COL))

print(f"Total records loaded: {len(raw_data):,}")

# ---- Encode Feature độc lập cho từng feature của từng Core ----
feature_encoders = {}
X_encoded = {core_name: [] for core_name in CORE_FEATURES.keys()}

for core_name, feat_list in CORE_FEATURES.items():
    arr = np.array(X_raw[core_name], dtype=object)
    encoded_cols = []
    
    for col_idx in range(arr.shape[1]):
        col_data = arr[:, col_idx]
        
        # Nếu cột chứa dữ liệu dạng chữ (string), tiến hành Label Encoding
        is_string = any(isinstance(val, str) for val in col_data)
        if is_string:
            le = LabelEncoder()
            encoded_col = le.fit_transform(col_data.astype(str)).astype(np.float32)
            feature_encoders[(core_name, col_idx)] = le
            print(f"  Encoded feature '{feat_list[col_idx]}' in Core '{core_name}'")
        else:
            encoded_col = col_data.astype(np.float32)
            
        encoded_cols.append(encoded_col)
    
    X_encoded[core_name] = np.stack(encoded_cols, axis=1)

# ---- Process Target (Regression) ----
y = np.array(y_raw, dtype=np.float32)
print(f"Target variable '{TARGET_COL}' loaded as continuous.")
print(f"Target distribution summary: Min={y.min()}, Max={y.max()}, Mean={y.mean():.4f}, Std={y.std():.4f}")

# ---- Train / Test / Val Split ----
indices = np.arange(len(y))

# 1. Tách tập test
train_val_idx, test_idx = train_test_split(
    indices, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

# 2. Tách tập train và validation từ tập còn lại
val_ratio = VAL_SIZE / (1 - TEST_SIZE)
train_idx, val_idx = train_test_split(
    train_val_idx, test_size=val_ratio, random_state=RANDOM_STATE
)

X_train_cores = {core: X_encoded[core][train_idx] for core in X_encoded}
X_val_cores   = {core: X_encoded[core][val_idx] for core in X_encoded}
X_test_cores  = {core: X_encoded[core][test_idx] for core in X_encoded}
y_train = y[train_idx]
y_val   = y[val_idx]
y_test  = y[test_idx]

# ---- Normalize độc lập từng Core (StandardScaler) ----
core_scalers = {}
core_cols_to_scale = {}

for core_name, feat_list in CORE_FEATURES.items():
    arr_train = X_train_cores[core_name]
    arr_val   = X_val_cores[core_name]
    arr_test  = X_test_cores[core_name]
    
    # Chỉ chuẩn hóa những cột nguyên bản là dữ liệu số (không bị mã hóa bởi LabelEncoder)
    cols_to_scale = []
    for col_idx in range(arr_train.shape[1]):
        if (core_name, col_idx) not in feature_encoders:
            cols_to_scale.append(col_idx)
            
    if cols_to_scale:
        scaler = StandardScaler()
        arr_train[:, cols_to_scale] = scaler.fit_transform(arr_train[:, cols_to_scale])
        arr_val[:, cols_to_scale]   = scaler.transform(arr_val[:, cols_to_scale])
        arr_test[:, cols_to_scale]  = scaler.transform(arr_test[:, cols_to_scale])
        core_scalers[core_name] = scaler
        core_cols_to_scale[core_name] = cols_to_scale
        print(f"  Normalized numerical features for Core '{core_name}': {[feat_list[i] for i in cols_to_scale]}")
        
    X_train_cores[core_name] = arr_train
    X_val_cores[core_name]   = arr_val
    X_test_cores[core_name]  = arr_test

# Định nghĩa số lượng chiều đầu vào cho từng nhánh
core_input_dims = {core: len(features) for core, features in CORE_FEATURES.items()}

print(f"\nTrain samples: {len(train_idx)} | Val samples: {len(val_idx)} | Test samples: {len(test_idx)}")
for core_name, dim in core_input_dims.items():
    print(f"  Branch '{core_name}' -> Input Dimension: {dim}")


  LOAD & PREPROCESS DATA (REGRESSION & ORDINAL LEVEL)
Total records loaded: 511
  Encoded feature 'Chronic Disease' in Core 'Clinical Risk Assessments'
  Encoded feature 'Smoking Status' in Core 'Mood & Behavioral Health'
  Encoded feature 'Gender' in Core 'Financial & Legal'
  Encoded feature 'Chronic Disease' in Core 'Financial & Legal'
Target variable 'Level' loaded as continuous.
Target distribution summary: Min=1.0, Max=3.0, Mean=1.9178, Std=0.8353
  Normalized numerical features for Core 'ADLs & IADLs': ['Age', 'Physical Activity']
  Normalized numerical features for Core 'Cognitive & Neurological Status': ['Sleep Duration', 'Stress Level']
  Normalized numerical features for Core 'Clinical Risk Assessments': ['BMI']
  Normalized numerical features for Core 'Mood & Behavioral Health': ['Stress Level', 'Alcohol Consumption']
  Normalized numerical features for Core 'Financial & Legal': ['Age']

Train samples: 331 | Val samples: 77 | Test samples: 103
  Branch 'ADLs & IADLs' -> Inp

In [6]:
print("\n" + "=" * 60)
print("  TRAINING & COMPARING COHESIVE PIPELINES (REGRESSION -> CLASSIFICATION)")
print("=" * 60)

def get_model_instance(name):
    if name == "Random Forest":
        return RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
    elif name == "Gradient Boosting":
        return GradientBoostingRegressor(n_estimators=150, learning_rate=0.1, max_depth=5, random_state=RANDOM_STATE)
    elif name == "XGBoost":
        return xgb.XGBRegressor(n_estimators=150, learning_rate=0.1, max_depth=6, random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)
    elif name == "LightGBM":
        return lgb.LGBMRegressor(n_estimators=150, learning_rate=0.1, max_depth=6, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1)
    return None

def build_meta_features(data, current_base_models):
    meta = []
    for core_name in core_input_dims:
        pred = current_base_models[core_name].predict(data[core_name])
        meta.append(pred.reshape(-1, 1))
    return np.hstack(meta)

pipeline_candidates = ["Random Forest", "Gradient Boosting", "XGBoost", "LightGBM"]
meta_results = []

for name in pipeline_candidates:
    print(f"\n{'-'*60}")
    print(f"  Training Stacked Pipeline: {name}")
    print(f"{'-'*60}")
    
    # 1. Huấn luyện 5 base models con tương ứng
    curr_base_models = {}
    print("  Base Models RMSE on Test Set:")
    for core_name in core_input_dims:
        model = get_model_instance(name)
        model.fit(X_train_cores[core_name], y_train)
        curr_base_models[core_name] = model
        
        pred = model.predict(X_test_cores[core_name])
        rmse = np.sqrt(mean_squared_error(y_test, pred))
        print(f"    {core_name:<15}: RMSE = {rmse:.4f}")
        
    # 2. Tạo tập đặc trưng Meta cho Train, Val và Test
    X_train_meta = build_meta_features(X_train_cores, curr_base_models)
    X_val_meta   = build_meta_features(X_val_cores, curr_base_models)
    X_test_meta  = build_meta_features(X_test_cores, curr_base_models)
    
    # 3. Huấn luyện mô hình Meta cùng thuật toán
    meta_model = get_model_instance(name)
    if name == "Random Forest":
        meta_model.set_params(n_estimators=300)
    meta_model.fit(X_train_meta, y_train)
    
    # 4. Dự đoán trên tập Val và Test thông qua mô hình Meta (Continuous predictions)
    y_val_pred_score  = meta_model.predict(X_val_meta)
    y_test_pred_score = meta_model.predict(X_test_meta)
    
    # 5. Chuyển đổi Regression score về Classification Level (1, 2, 3)
    y_val_pred_class  = np.array([score_to_level(x) for x in y_val_pred_score])
    y_test_pred_class = np.array([score_to_level(x) for x in y_test_pred_score])
    
    y_val_class  = y_val.astype(int)
    y_test_class = y_test.astype(int)
    
    # 6. Tính toán các độ đo Regression
    val_rmse  = np.sqrt(mean_squared_error(y_val, y_val_pred_score))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_score))
    test_mae  = mean_absolute_error(y_test, y_test_pred_score)
    test_r2   = r2_score(y_test, y_test_pred_score)
    
    # 7. Tính toán các độ đo Classification (sau khi Mapping)
    val_acc      = accuracy_score(y_val_class, y_val_pred_class)
    test_acc     = accuracy_score(y_test_class, y_test_pred_class)
    test_macro_f = f1_score(y_test_class, y_test_pred_class, average="macro", zero_division=0)
    test_weight_f = f1_score(y_test_class, y_test_pred_class, average="weighted", zero_division=0)
    
    meta_results.append({
        "pipeline_name": name,
        "base_models": curr_base_models,
        "meta_model": meta_model,
        "predictions_score": y_test_pred_score,
        "predictions_class": y_test_pred_class,
        "val_rmse": val_rmse,
        "test_rmse": test_rmse,
        "test_mae": test_mae,
        "test_r2": test_r2,
        "val_acc": val_acc,
        "test_acc": test_acc,
        "test_macro_f1": test_macro_f,
        "test_weighted_f1": test_weight_f
    })
    
    print(f"\n  [Meta Model: {name}]")
    print(f"    Regression     -> Val RMSE: {val_rmse:.4f} | Test RMSE: {test_rmse:.4f} | Test R2: {test_r2:.4f}")
    print(f"    Classification -> Val Acc : {val_acc:.4f} | Test Acc : {test_acc:.4f} | Test Macro F1: {test_macro_f:.4f}")



  TRAINING & COMPARING COHESIVE PIPELINES (REGRESSION -> CLASSIFICATION)

------------------------------------------------------------
  Training Stacked Pipeline: Random Forest
------------------------------------------------------------
  Base Models RMSE on Test Set:
    ADLs & IADLs   : RMSE = 0.9617
    Cognitive & Neurological Status: RMSE = 0.8664
    Clinical Risk Assessments: RMSE = 0.9763
    Mood & Behavioral Health: RMSE = 0.9153
    Financial & Legal: RMSE = 0.9529

  [Meta Model: Random Forest]
    Regression     -> Val RMSE: 0.9658 | Test RMSE: 1.0489 | Test R2: -0.7180
    Classification -> Val Acc : 0.3636 | Test Acc : 0.3107 | Test Macro F1: 0.2585

------------------------------------------------------------
  Training Stacked Pipeline: Gradient Boosting
------------------------------------------------------------
  Base Models RMSE on Test Set:
    ADLs & IADLs   : RMSE = 1.0673
    Cognitive & Neurological Status: RMSE = 0.9105
    Clinical Risk Assessments: RMSE 

In [7]:
print("\n" + "=" * 85)
print("  PIPELINES COMPARISON SUMMARY")
print("=" * 85)

comparison_df = pd.DataFrame(meta_results)[[
    "pipeline_name", "val_rmse", "test_rmse", "test_mae", "test_r2", "val_acc", "test_acc", "test_macro_f1"
]].sort_values("test_rmse", ascending=True)

print(comparison_df.to_string(index=False))

# Lựa chọn Pipeline tốt nhất dựa trên RMSE
best_idx = next(i for i, r in enumerate(meta_results) if r["pipeline_name"] == comparison_df.iloc[0]["pipeline_name"])
best_pipeline = meta_results[best_idx]
best_name = best_pipeline["pipeline_name"]

print(f"\n=== Best Pipeline: {best_name} ===")

print("\n--- Regression Metrics ---")
print(f"   Validation RMSE: {best_pipeline['val_rmse']:.4f}")
print(f"   Test RMSE      : {best_pipeline['test_rmse']:.4f}")
print(f"   Test MAE       : {best_pipeline['test_mae']:.4f}")
print(f"   Test R2-Score  : {best_pipeline['test_r2']:.4f}")

print("\n--- Classification Metrics (After Threshold Mapping) ---")
print(f"   Validation Acc : {best_pipeline['val_acc']:.4f}")
print(f"   Test Accuracy  : {best_pipeline['test_acc']:.4f}")
print(f"   Test Macro F1  : {best_pipeline['test_macro_f1']:.4f}")
print(f"   Test Weighted F1: {best_pipeline['test_weighted_f1']:.4f}")

y_test_class = y_test.astype(int)
print(f"\nDetailed Classification Report — {best_name}:")
print(classification_report(y_test_class, best_pipeline["predictions_class"], target_names=["1", "2", "3"]))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_class, best_pipeline["predictions_class"]))



  PIPELINES COMPARISON SUMMARY
    pipeline_name  val_rmse  test_rmse  test_mae   test_r2  val_acc  test_acc  test_macro_f1
    Random Forest  0.965758   1.048896  0.861294 -0.717963 0.363636  0.310680       0.258529
         LightGBM  0.987228   1.069539  0.895039 -0.786249 0.389610  0.349515       0.317630
          XGBoost  0.986212   1.087635  0.884189 -0.847206 0.298701  0.330097       0.299944
Gradient Boosting  1.044888   1.140063  0.858741 -1.029583 0.324675  0.388350       0.381771

=== Best Pipeline: Random Forest ===

--- Regression Metrics ---
   Validation RMSE: 0.9658
   Test RMSE      : 1.0489
   Test MAE       : 0.8613
   Test R2-Score  : -0.7180

--- Classification Metrics (After Threshold Mapping) ---
   Validation Acc : 0.3636
   Test Accuracy  : 0.3107
   Test Macro F1  : 0.2585
   Test Weighted F1: 0.2682

Detailed Classification Report — Random Forest:
              precision    recall  f1-score   support

           1       0.26      0.29      0.27        34
   

In [8]:
print("\n" + "=" * 60)
print("  SAVING ALL TRAINED PIPELINES")
print("=" * 60)

for r in meta_results:
    model_name = r["pipeline_name"]
    file_name = f"{model_name.lower().replace(' ', '_')}_regression_pipeline.pkl"
    file_path = os.path.join(MODEL_SAVE_DIR, file_name)
    
    pipeline_data = {
        "pipeline_name": model_name,
        "base_models": r["base_models"],
        "meta_model": r["meta_model"],
        "feature_encoders": feature_encoders,
        "core_scalers": core_scalers,
        "core_cols_to_scale": core_cols_to_scale,
        "target_encoder": None,
        "core_features": CORE_FEATURES,
        "score_thresholds": [1.5, 2.5]
    }
    
    joblib.dump(pipeline_data, file_path)
    print(f"Saved pipeline '{model_name}' to '{file_path}'")



  SAVING ALL TRAINED PIPELINES
Saved pipeline 'Random Forest' to 'D:\2026\MockProject_062026_NhomAI\model\random_forest_regression_pipeline.pkl'
Saved pipeline 'Gradient Boosting' to 'D:\2026\MockProject_062026_NhomAI\model\gradient_boosting_regression_pipeline.pkl'
Saved pipeline 'XGBoost' to 'D:\2026\MockProject_062026_NhomAI\model\xgboost_regression_pipeline.pkl'
Saved pipeline 'LightGBM' to 'D:\2026\MockProject_062026_NhomAI\model\lightgbm_regression_pipeline.pkl'
